In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
import time

options = Options()
options.add_argument("--disable-blink-features=AutomationControlled")
options.add_experimental_option("excludeSwitches", ["enable-automation"])
options.add_experimental_option("useAutomationExtension", False)

driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

# Primero navega a .jgi.doe.gov para poder agregar cookies con ese dominio
driver.get("https://jgi.doe.gov")
time.sleep(20)

# Agregar cookies con dominio .jgi.doe.gov
cookies_jgi = [
    {"name": "cf_clearance", "value": "xmzhcmqVOLBmbf0UczY.2.1.1-F7Bl5od6W1rzcjPGzA9IiL0Y9cuDq2wp89ChfN_38ZKeG7UTJEeFEqE9ZIWIRCM6t6g36aiSqvAEbTvlvQTtFZ0x54pz9I.7VHs1XWIHU0f5N8cw4HEgRojhIHOcSlepYFdc0h2PJZRRZwVkHppHB4kuUm3utkqvFSOA8TuCIBTkpTpGqmwy7_r8R.Tbd31dvMWgYLmVixe4_FM91YaqVgC.pP8qWK9rVw2rP7VBxhGfaGLLDRIXkt.0X25qF0EiMFwY7KRhP0d3Dupd1y5XSSk7zYlkUtMyHNC8ShHGD13ZhdxxojIYBjiFT4BuRk6Q.2NdmpiQTN3voVeLrDySStD7zZ3PFrIg_lrbbxALUeBzaZ5BBcBxSajGDi.1xP7FhOuOJZqTQ9topMO3Pjf18yvRF4JNB3wXDTBiFXsFWnEc1moYAFu.QlBYpk1faV1KcSSSgsac4LxgNi5BqEz_aEyL4Rxag.EtFzi35N7yb58RfqheUiflRGEiix5Ue6nrMSm9fr2Hguj7OfKyhP9azA", "domain": ".jgi.doe.gov"},
    {"name": "_cf_bm", "value": "cyBkjE7wipcad9wZIS6OwYsB0vnLe1.0.1.1-N2dIt3gzBYcpAyOt.B5Cs2qcvzAh0_YJ4w5jpzRD2cj3z0rN6bOanlwMvY0VXGFV8ygA3Ll9NFRmU68GdqV4SdDyn.4FBoHhhfH8weaX8nPPUtGn2A1hpTfmhUpTXzKE", "domain": ".jgi.doe.gov"},  # reemplaza valor
]
for cookie in cookies_jgi:
    driver.add_cookie(cookie)

# Ahora navega a mycocosm.jgi.doe.gov para agregar la cookie de sesión
driver.get("https://mycocosm.jgi.doe.gov")
time.sleep(20)

# Agregar JSESSIONID para mycocosm
driver.add_cookie({"name": "JSESSIONID", "value": "mycocosm.jgi.doe.gov", "domain": "mycocosm.jgi.doe.gov"})

# Finalmente, ir a la URL destino
driver.get("https://mycocosm.jgi.doe.gov/cgi-bin/metapathways?db=TriharM10_1")
time.sleep(10)

html = driver.page_source
with open("TriharM10_1_selenium_cookies.html", "w", encoding="utf-8") as f:
    f.write(html)

print("✅ HTML guardado con Selenium.")
driver.quit()

In [5]:

import os
import re
import time
import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
from bs4 import BeautifulSoup
# ------------------------------------------------------------------
# 1. CONFIGURACIÓN INICIAL (cookies y navegador)
# ------------------------------------------------------------------
options = Options()
options.add_argument("--disable-blink-features=AutomationControlled")
options.add_experimental_option("excludeSwitches", ["enable-automation"])
options.add_experimental_option("useAutomationExtension", False)

driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

# -- Navegar a jgi.doe.gov para establecer dominio y agregar cookies --
driver.get("https://jgi.doe.gov")
time.sleep(5)

# --- COOKIES (las que ya te funcionaron) ---
cookies_jgi = [
    {"name": "cf_clearance", 
     "value": "944sET97JxEreeB4fFSsMHtLaQWs1rhgw8FGi0jDC8A-1787374497-1.2.1.1-_wL6cFdeFVpiml2m0zEmcGgXnc2E0Poh.Tnog_MneLHRAwyHfhn2sl.fQc0CjTxldzrFkBuT.s6ICibBsL3un0rrKgjN0PwF79bVfvZ2kmvyGuYjnHqrqgXaU3.vHNyEe_Nl4O2nu9jIF1AnxWABICJaiuINjsi0Pw1dpacEAiLQSvZiqpJpspC1l5WDujDyJAiT13kYJShBFysfNHfaz9HG2RTzBVPJB_RNOw6CTc4LUwCMUDEODXbucuXPalTW8Vinb.JVpqYh6bmwIjQoWjxS.voXkrpbNe6Id7wmCqROe1kR2..D97iY0qTyTT442vI7wZRO9IBOrJzLp7pgrYWRSmdlP68ovkxmzjxq3nhlwdGPWoM8s2GvP6xCzbkk7gE0dQAn_CGvjjOhDS2kXiBaWr7w8Rd5n0.2wZ5pVB2DPukhQLSHjxqhYbw.U6hd6DR0igOrZj4BDWpXut8QiafBc0gZ5yUmWX1vdB692O.rWUePSVNAOjucaTCd3b37BhH0jLoae2aBdMPoWShV3Q", 
     "domain": ".jgi.doe.gov"},
     
    {"name": "_cf_bm", 
     "value": "zQQ8oXvBkcFmFRqiHfgbBYIj.zrKYq..QAUy2pN1yI0-1787382384.0674005-1.0.1.1-BsWGfEcniPI82N.oM1h8eON6rzvKozsJ75UlBBwOtnXfSRtxJ_CFb65dMNRloW2LWpEC0Z3dFtibhLkqZPR8a.6C1HzbtyUnjMObXO8CvN6fJ.EkpFhBydicNgmyZvUj", 
     "domain": ".jgi.doe.gov"},
]

for cookie in cookies_jgi:
    driver.add_cookie(cookie)

# -- Navegar a mycocosm.jgi.doe.gov para agregar JSESSIONID --
driver.get("https://mycocosm.jgi.doe.gov")
time.sleep(5)

driver.add_cookie({
    "name": "JSESSIONID", 
    "value": "B55FBBCCBF20DF158E3E1193612311A3", 
    "domain": "mycocosm.jgi.doe.gov"
})

# -- Ir a la página principal de rutas --
target_url = "https://mycocosm.jgi.doe.gov/cgi-bin/metapathways?db=TriharM10_1"
driver.get(target_url)
time.sleep(10)  # esperar carga completa

# ------------------------------------------------------------------
# 2. EXTRAER ENLACES DE RUTAS DESDE LA PÁGINA PRINCIPAL
# ------------------------------------------------------------------
html_principal = driver.page_source

# Guardar la página principal para depuración
os.makedirs("API", exist_ok=True)
with open("API/TriharM10_1_principal.html", "w", encoding="utf-8") as f:
    f.write(html_principal)
print("✅ Página principal guardada en API/ (revisa este archivo para ver si contiene los enlaces).")

# Parsear con BeautifulSoup
soup = BeautifulSoup(html_principal, 'html.parser')

# Buscar todos los enlaces que contengan 'metapathways?map='
enlaces = []
for a in soup.find_all('a', href=True):
    href = a['href']
    if 'metapathways?map=' in href:
        url_completa = 'https://mycocosm.jgi.doe.gov/cgi-bin/' + href if href.startswith('metapathways?') else href
        nombre = a.get_text(strip=True)
        enlaces.append((url_completa, nombre))

# Si no se encontraron con el primer método, intentar con un filtro más amplio
if not enlaces:
    print("⚠️ No se encontraron enlaces con 'metapathways?map='. Intentando con 'map='...")
    for a in soup.find_all('a', href=True):
        href = a['href']
        if 'map=' in href and 'metapathways' in href:
            url_completa = 'https://mycocosm.jgi.doe.gov/cgi-bin/' + href if href.startswith('metapathways?') else href
            nombre = a.get_text(strip=True)
            enlaces.append((url_completa, nombre))

print(f"🔗 Se encontraron {len(enlaces)} enlaces de rutas metabólicas.")

# Mostrar algunos enlaces para depuración
if enlaces:
    print("📌 Ejemplos de enlaces encontrados:")
    for i, (url, nombre) in enumerate(enlaces[:5]):
        print(f"   {i+1}. {nombre} -> {url}")
else:
    print("❌ No se encontraron enlaces. Verifica que el HTML guardado contenga los enlaces.")
    print("   Puedes abrir 'API/TriharM10_1_principal.html' en un navegador para inspeccionarlo.")
    driver.quit()
    exit()

# ------------------------------------------------------------------
# 3. SEPARAR ENLACES POR TIPO: NOMBRES (rutas) y NÚMEROS (modelos)
# ------------------------------------------------------------------
enlaces_rutas = []      # para nombres de vía (no numéricos)
enlaces_modelos = []    # para números (ej. "41", "104")

for url, nombre in enlaces:
    if nombre.isdigit():
        # Es un número → lo guardamos para la descarga de modelos
        enlaces_modelos.append((url, nombre))
    else:
        # Es un nombre de vía → descarga normal
        enlaces_rutas.append((url, nombre))

print(f"📊 Resumen:")
print(f"   - Rutas metabólicas (nombres): {len(enlaces_rutas)}")
print(f"   - Enlaces numéricos (modelos): {len(enlaces_modelos)}")

# ------------------------------------------------------------------
# 4. DESCARGAR PÁGINAS DE RUTAS (SOLO SI NO EXISTEN)
# ------------------------------------------------------------------
print("\n📥 Descargando páginas de rutas metabólicas...")

descargados_rutas = 0
omitidos_rutas = 0

for i, (url, nombre) in enumerate(enlaces_rutas, 1):
    nombre_limpio = nombre.replace('/', '_').replace(' ', '_').replace(',', '')
    if len(nombre_limpio) > 50:
        nombre_limpio = nombre_limpio[:50]
    if 'map=' in url:
        map_id = url.split('map=')[1].split('&')[0]
    else:
        map_id = f"ruta_{i}"
    archivo = f"API/{map_id}_{nombre_limpio}.html"
    
    if os.path.exists(archivo):
        print(f"⏭️ [{i}/{len(enlaces_rutas)}] Omitiendo {nombre} ({map_id}) - archivo ya existe.")
        omitidos_rutas += 1
        continue
    
    print(f"\n📥 [{i}/{len(enlaces_rutas)}] Descargando ruta: {nombre} ({map_id})")
    try:
        driver.get(url)
        WebDriverWait(driver, 15).until(EC.presence_of_element_located((By.TAG_NAME, "body")))
        time.sleep(3)
        
        html_ruta = driver.page_source
        with open(archivo, "w", encoding="utf-8") as f:
            f.write(html_ruta)
        print(f"   ✅ Guardado: {archivo}")
        descargados_rutas += 1
        time.sleep(1)
    except Exception as e:
        print(f"   ❌ Error al descargar {url}: {e}")

print(f"\n✅ Rutas completadas. Descargados: {descargados_rutas}, Omitidos: {omitidos_rutas}")

# ------------------------------------------------------------------
# 5. DESCARGAR PÁGINAS DE SUBSYSTEMS (MODELOS) A PARTIR DE LOS NÚMEROS
# ------------------------------------------------------------------
print("\n📥 Descargando páginas de subsystems (modelos)...")

# Crear subcarpeta para subsystems
os.makedirs("API/subsystems", exist_ok=True)

descargados_subs = 0
omitidos_subs = 0

for i, (url, numero) in enumerate(enlaces_modelos, 1):
    # Extraer map_id de la URL
    if 'map=' in url:
        map_id = url.split('map=')[1].split('&')[0]
    else:
        map_id = f"sub_{i}"
    
    # Construir la URL con models=1
    # Ejemplo: https://mycocosm.jgi.doe.gov/cgi-bin/metapathways?db=TriharM10_1&map=MAP00330&models=1&col=&batchId=TriharM10_1:1
    url_subsystem = f"https://mycocosm.jgi.doe.gov/cgi-bin/metapathways?db=TriharM10_1&map={map_id}&models=1&col=&batchId=TriharM10_1:1"
    
    archivo = f"API/subsystems/{map_id}_models.html"
    
    if os.path.exists(archivo):
        print(f"⏭️ [{i}/{len(enlaces_modelos)}] Omitiendo {map_id} (models=1) - archivo ya existe.")
        omitidos_subs += 1
        continue
    
    print(f"\n📥 [{i}/{len(enlaces_modelos)}] Descargando subsystems para {map_id} (modelos: {numero})")
    try:
        driver.get(url_subsystem)
        WebDriverWait(driver, 15).until(EC.presence_of_element_located((By.TAG_NAME, "body")))
        time.sleep(3)
        
        html_subs = driver.page_source
        with open(archivo, "w", encoding="utf-8") as f:
            f.write(html_subs)
        print(f"   ✅ Guardado: {archivo}")
        descargados_subs += 1
        time.sleep(1)
    except Exception as e:
        print(f"   ❌ Error al descargar {url_subsystem}: {e}")

print(f"\n✅ Subsystems completados. Descargados: {descargados_subs}, Omitidos: {omitidos_subs}")

# ------------------------------------------------------------------
# 6. PROCESAR HTML DE SUBSYSTEMS: EXTRAER PROTEIN ID Y EC NUM
# ------------------------------------------------------------------
print("\n🔍 Procesando archivos de subsystems para extraer Protein ID y EC Num...")

# Diccionario para almacenar: protein_id -> ec_num (de cada archivo)
protein_ec_map = {}

# Listar todos los archivos HTML en API/subsystems/
subsystem_files = [f for f in os.listdir("API/subsystems") if f.endswith(".html")]
total_archivos = len(subsystem_files)

for idx, filename in enumerate(subsystem_files, 1):
    print(f"\n📄 [{idx}/{total_archivos}] Procesando archivo: {filename}")
    
    filepath = os.path.join("API/subsystems", filename)
    with open(filepath, "r", encoding="utf-8") as f:
        html_content = f.read()
    
    soup = BeautifulSoup(html_content, 'html.parser')
    
    # Buscar la tabla que contiene los datos
    tables = soup.find_all('table')
    if not tables:
        print(f"   ⚠️ No se encontraron tablas en {filename}")
        continue
    
    archivo_protein_ec = {}  # para contar cuántos se extraen de este archivo
    
    for table in tables:
        rows = table.find_all('tr')
        if len(rows) < 2:
            continue
        
        # Buscar encabezados
        header_row = rows[0]
        header_cells = header_row.find_all(['th', 'td'])
        header_texts = [cell.get_text(strip=True) for cell in header_cells]
        
        try:
            idx_protein = header_texts.index("Protein ID")
            idx_ec = header_texts.index("EC Num")
        except ValueError:
            try:
                idx_protein = next(i for i, t in enumerate(header_texts) if "protein" in t.lower())
                idx_ec = next(i for i, t in enumerate(header_texts) if "ec" in t.lower() and "num" in t.lower())
            except StopIteration:
                continue
        
        # Procesar filas de datos
        for row in rows[1:]:
            cells = row.find_all(['th', 'td'])
            if len(cells) <= max(idx_protein, idx_ec):
                continue
            
            protein_id = cells[idx_protein].get_text(strip=True)
            ec_num = cells[idx_ec].get_text(strip=True)
            
            if protein_id and ec_num and re.match(r'^\d+\.\d+\.\d+\.?-?\d*$', ec_num):
                # Guardar en el diccionario principal
                protein_ec_map[protein_id] = ec_num
                # Guardar también en el dict del archivo para contar
                archivo_protein_ec[protein_id] = ec_num
        
        # Si encontramos datos, salir del bucle de tablas
        if archivo_protein_ec:
            break
    
    # Mostrar cuántos se extrajeron de este archivo
    if archivo_protein_ec:
        print(f"   ✅ Extraídos {len(archivo_protein_ec)} pares de este archivo.")
    else:
        print(f"   ℹ️ No se encontraron pares válidos en este archivo.")

print(f"\n✅ Total extraídos: {len(protein_ec_map)} pares Protein ID -> EC Num desde los {total_archivos} archivos procesados.")

# ------------------------------------------------------------------
# 7. CARGAR CSV Y BUSCAR MATCHES
# ------------------------------------------------------------------
print("\n📂 Cargando genesModelTemp2.csv...")
try:
    import pandas as pd
    df_genes = pd.read_csv("genesModelTemp2.csv")
    print(f"✅ CSV cargado con {len(df_genes)} registros.")
except FileNotFoundError:
    print("❌ Error: No se encontró el archivo 'genesModelTemp2.csv'.")
    print("   Asegúrate de que el archivo esté en el mismo directorio que el script.")
    driver.quit()
    exit()
except Exception as e:
    print(f"❌ Error al leer el CSV: {e}")
    driver.quit()
    exit()

# Crear un conjunto de Protein IDs del CSV para búsqueda rápida
protein_ids_csv = set(df_genes['ProteinID'].astype(str).values)

# Buscar matches entre los Protein IDs extraídos y los del CSV
matches = {}
for protein_id, ec_num in protein_ec_map.items():
    if protein_id in protein_ids_csv:
        matches[protein_id] = ec_num

print(f"🔗 Encontrados {len(matches)} matches entre Protein IDs y el CSV.")

if not matches:
    print("⏹️ No hay matches. No se descargarán páginas de EC.")
else:
    # ------------------------------------------------------------------
    # 8. DESCARGAR PÁGINAS DE EC PARA LOS MATCHES
    # ------------------------------------------------------------------
    print("\n📥 Descargando páginas de EC para los matches...")
    
    # Crear subcarpeta para EC
    os.makedirs("API/subsystems/ec", exist_ok=True)
    
    descargados_ec = 0
    omitidos_ec = 0
    
    for protein_id, ec_num in matches.items():
        # Dividir el EC num en sus componentes (ej. 1.2.3.4 -> ec1=1, ec2=2, ec3=3, ec4=4)
        ec_parts = ec_num.split('.')
        # Asegurar que tenemos al menos 3 partes, y hasta 4
        ec1 = ec_parts[0] if len(ec_parts) > 0 else ''
        ec2 = ec_parts[1] if len(ec_parts) > 1 else ''
        ec3 = ec_parts[2] if len(ec_parts) > 2 else ''
        ec4 = ec_parts[3] if len(ec_parts) > 3 else ''
        
        # Construir la URL de búsqueda KEGG
        url_ec = f"https://mycocosm.jgi.doe.gov/cgi-bin/searchKEGG?field1=EC+Number&ec1={ec1}&ec2={ec2}&ec3={ec3}&ec4={ec4}&species=TriharM10_1&info=KEGG&type=KEGG&batchId=TriharM10_1:1"
        
        # Nombre del archivo: protein_id_ec_num.html
        archivo_ec = f"API/subsystems/ec/{protein_id}_{ec_num}.html"
        
        # Verificar si ya existe
        if os.path.exists(archivo_ec):
            print(f"   ⏭️ Omitiendo EC {ec_num} (Protein {protein_id}) - archivo ya existe.")
            omitidos_ec += 1
            continue
        
        print(f"   📥 Descargando EC {ec_num} para Protein {protein_id}...")
        try:
            driver.get(url_ec)
            WebDriverWait(driver, 15).until(EC.presence_of_element_located((By.TAG_NAME, "body")))
            time.sleep(3)
            
            html_ec = driver.page_source
            with open(archivo_ec, "w", encoding="utf-8") as f:
                f.write(html_ec)
            print(f"      ✅ Guardado: {archivo_ec}")
            descargados_ec += 1
            time.sleep(1)
        except Exception as e:
            print(f"      ❌ Error al descargar {url_ec}: {e}")
    
    print(f"\n✅ EC completados. Descargados: {descargados_ec}, Omitidos: {omitidos_ec}")

# ------------------------------------------------------------------
# 9. RESUMEN FINAL (modificado)
# ------------------------------------------------------------------
print("\n🎉 ¡Descarga completada!")
print(f"   📁 Rutas metabólicas: API/")
print(f"      Descargados: {descargados_rutas}, Omitidos: {omitidos_rutas}")
print(f"   📁 Subsystems (models=1): API/subsystems/")
print(f"      Descargados: {descargados_subs}, Omitidos: {omitidos_subs}")
if 'matches' in locals():
    print(f"   📁 EC matches (descargados): API/subsystems/ec/")
    print(f"      Descargados: {descargados_ec}, Omitidos: {omitidos_ec}")

driver.quit()


✅ Página principal guardada en API/ (revisa este archivo para ver si contiene los enlaces).
⚠️ No se encontraron enlaces con 'metapathways?map='. Intentando con 'map='...
🔗 Se encontraron 314 enlaces de rutas metabólicas.
📌 Ejemplos de enlaces encontrados:
   1. Alanine, aspartate and glutamate metabolism -> https://mycocosm.jgi.doe.gov/cgi-bin/metapathways?db=TriharM10_1&map=MAP00250&batchId=TriharM10_1:1
   2. 41 -> https://mycocosm.jgi.doe.gov/cgi-bin/metapathways?db=TriharM10_1&map=MAP00250&batchId=TriharM10_1:1
   3. Arginine and proline metabolism -> https://mycocosm.jgi.doe.gov/cgi-bin/metapathways?db=TriharM10_1&map=MAP00330&batchId=TriharM10_1:1
   4. 104 -> https://mycocosm.jgi.doe.gov/cgi-bin/metapathways?db=TriharM10_1&map=MAP00330&batchId=TriharM10_1:1
   5. Cysteine and methionine metabolism -> https://mycocosm.jgi.doe.gov/cgi-bin/metapathways?db=TriharM10_1&map=MAP00270&batchId=TriharM10_1:1
📊 Resumen:
   - Rutas metabólicas (nombres): 164
   - Enlaces numéricos (modelos

In [1]:
import os
import re
import time
import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
from bs4 import BeautifulSoup
# ------------------------------------------------------------------
# 1. CONFIGURACIÓN INICIAL (cookies y navegador)
# ------------------------------------------------------------------
options = Options()
options.add_argument("--disable-blink-features=AutomationControlled")
options.add_experimental_option("excludeSwitches", ["enable-automation"])
options.add_experimental_option("useAutomationExtension", False)

driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

# -- Navegar a jgi.doe.gov para establecer dominio y agregar cookies --
driver.get("https://jgi.doe.gov")
time.sleep(5)

# --- COOKIES (las que ya te funcionaron) ---
cookies_jgi = [
    {"name": "cf_clearance", 
     "value": "VcppgqqdWhEuyQdnGIiA6M5XkLmHQeir8ngrUAbIp5s-1788020216-1.2.1.1-u2VzFA1a049Cw5XfXczII28c5V5xv2U1_W3C9sJefY2B.xrcbxALPbmlA8eM00ZfdwIXZCNiGhcNPA8z3uarhnhb8OXA9vCeeGlr9Krd6cHb_5aiXYGdeBPN95GBqXJwMyyhrYFGpbscRaUXaHov4m6f5qk0zpkLgYyQB27G4HptLdy.RtSdb23xKTAZNy.kEFM7hlyvf3pO.h2bWkapBYB_KE9NbOzil5Ln_I50ILY57xsqWwk7f3CqFkYUJjxEsjKyr2dLP7K.JIav1wIVTFVPmxiWEem5E4SHdjHwJyzGRiwltYIGfUU3PvQvMu9m6SkK92BJc2YAdiAbgIEPy1.A9G91S5fO2mzmMdgkz0STNHBL4YQND0yuatiSBtRMTETz.cAXdOGcIBxyXtzdtAuEQ_2bjcVolRsPjhDX8HZhRHl0WwlQdNb7WZpORvD6dfNMf_89x39NqYiAYf4ry3GvYiAnJXShwfBVD_Rtfc8_FllkiH0IVzTbUZs0Fo3I1GvKTlJyhIfl3GU3jsM1lA", 
     "domain": ".jgi.doe.gov"},
     
    {"name": "_cf_bm", 
     "value": "vumsIksY7RK9r2SpMspyHSGOvq3lJeJnzeRneXCWl7Y-1788098702.3404732-1.0.1.1-zoZO7o.z.7u7ld1_bq4gAmLaGBFiDS4kJz6IONX._X1YMjq9Fa7P1_4FaEuhe8BFTy9iYSEyD4kcaf6eVwhYk4HCG4IvZdnRBY8kESpjUCxI1sx_bsfcaDUMo.HhIkpw", 
     "domain": ".jgi.doe.gov"},
]

for cookie in cookies_jgi:
    driver.add_cookie(cookie)

# -- Navegar a mycocosm.jgi.doe.gov para agregar JSESSIONID --
driver.get("https://mycocosm.jgi.doe.gov")
time.sleep(5)

driver.add_cookie({
    "name": "JSESSIONID", 
    "value": "F21B2B01135C3FA4CF3711B800ACBBBA", 
    "domain": "mycocosm.jgi.doe.gov"
})

# -- Ir a la página principal de rutas --
target_url = "https://mycocosm.jgi.doe.gov/cgi-bin/metapathways?db=TriharM10_1"
driver.get(target_url)
time.sleep(10)  # esperar carga completa

# ------------------------------------------------------------------
# 2. EXTRAER ENLACES DE RUTAS DESDE LA PÁGINA PRINCIPAL
# ------------------------------------------------------------------
html_principal = driver.page_source

# Guardar la página principal para depuración
os.makedirs("API", exist_ok=True)
with open("API/TriharM10_1_principal.html", "w", encoding="utf-8") as f:
    f.write(html_principal)
print("✅ Página principal guardada en API/ (revisa este archivo para ver si contiene los enlaces).")

# Parsear con BeautifulSoup
soup = BeautifulSoup(html_principal, 'html.parser')

# Buscar todos los enlaces que contengan 'metapathways?map='
enlaces = []
for a in soup.find_all('a', href=True):
    href = a['href']
    if 'metapathways?map=' in href:
        url_completa = 'https://mycocosm.jgi.doe.gov/cgi-bin/' + href if href.startswith('metapathways?') else href
        nombre = a.get_text(strip=True)
        enlaces.append((url_completa, nombre))

# Si no se encontraron con el primer método, intentar con un filtro más amplio
if not enlaces:
    print("⚠️ No se encontraron enlaces con 'metapathways?map='. Intentando con 'map='...")
    for a in soup.find_all('a', href=True):
        href = a['href']
        if 'map=' in href and 'metapathways' in href:
            url_completa = 'https://mycocosm.jgi.doe.gov/cgi-bin/' + href if href.startswith('metapathways?') else href
            nombre = a.get_text(strip=True)
            enlaces.append((url_completa, nombre))

print(f"🔗 Se encontraron {len(enlaces)} enlaces de rutas metabólicas.")

# Mostrar algunos enlaces para depuración
if enlaces:
    print("📌 Ejemplos de enlaces encontrados:")
    for i, (url, nombre) in enumerate(enlaces[:5]):
        print(f"   {i+1}. {nombre} -> {url}")
else:
    print("❌ No se encontraron enlaces. Verifica que el HTML guardado contenga los enlaces.")
    print("   Puedes abrir 'API/TriharM10_1_principal.html' en un navegador para inspeccionarlo.")
    driver.quit()
    exit()

# ------------------------------------------------------------------
# 3. SEPARAR ENLACES POR TIPO: NOMBRES (rutas) y NÚMEROS (modelos)
# ------------------------------------------------------------------
enlaces_rutas = []      # para nombres de vía (no numéricos)
enlaces_modelos = []    # para números (ej. "41", "104")

for url, nombre in enlaces:
    if nombre.isdigit():
        # Es un número → lo guardamos para la descarga de modelos
        enlaces_modelos.append((url, nombre))
    else:
        # Es un nombre de vía → descarga normal
        enlaces_rutas.append((url, nombre))

print(f"📊 Resumen:")
print(f"   - Rutas metabólicas (nombres): {len(enlaces_rutas)}")
print(f"   - Enlaces numéricos (modelos): {len(enlaces_modelos)}")

# ------------------------------------------------------------------
# 4. DESCARGAR PÁGINAS DE RUTAS (SOLO SI NO EXISTEN)
# ------------------------------------------------------------------
print("\n📥 Descargando páginas de rutas metabólicas...")

descargados_rutas = 0
omitidos_rutas = 0

for i, (url, nombre) in enumerate(enlaces_rutas, 1):
    nombre_limpio = nombre.replace('/', '_').replace(' ', '_').replace(',', '')
    if len(nombre_limpio) > 50:
        nombre_limpio = nombre_limpio[:50]
    if 'map=' in url:
        map_id = url.split('map=')[1].split('&')[0]
    else:
        map_id = f"ruta_{i}"
    archivo = f"API/{map_id}_{nombre_limpio}.html"
    
    if os.path.exists(archivo):
        print(f"⏭️ [{i}/{len(enlaces_rutas)}] Omitiendo {nombre} ({map_id}) - archivo ya existe.")
        omitidos_rutas += 1
        continue
    
    print(f"\n📥 [{i}/{len(enlaces_rutas)}] Descargando ruta: {nombre} ({map_id})")
    try:
        driver.get(url)
        WebDriverWait(driver, 15).until(EC.presence_of_element_located((By.TAG_NAME, "body")))
        time.sleep(3)
        
        html_ruta = driver.page_source
        with open(archivo, "w", encoding="utf-8") as f:
            f.write(html_ruta)
        print(f"   ✅ Guardado: {archivo}")
        descargados_rutas += 1
        time.sleep(1)
    except Exception as e:
        print(f"   ❌ Error al descargar {url}: {e}")

print(f"\n✅ Rutas completadas. Descargados: {descargados_rutas}, Omitidos: {omitidos_rutas}")

# ------------------------------------------------------------------
# 5. DESCARGAR PÁGINAS DE SUBSYSTEMS (MODELOS) A PARTIR DE LOS NÚMEROS
# ------------------------------------------------------------------
print("\n📥 Descargando páginas de subsystems (modelos)...")

# Crear subcarpeta para subsystems
os.makedirs("API/subsystems", exist_ok=True)

descargados_subs = 0
omitidos_subs = 0

for i, (url, numero) in enumerate(enlaces_modelos, 1):
    # Extraer map_id de la URL
    if 'map=' in url:
        map_id = url.split('map=')[1].split('&')[0]
    else:
        map_id = f"sub_{i}"
    
    # Construir la URL con models=1
    # Ejemplo: https://mycocosm.jgi.doe.gov/cgi-bin/metapathways?db=TriharM10_1&map=MAP00330&models=1&col=&batchId=TriharM10_1:1
    url_subsystem = f"https://mycocosm.jgi.doe.gov/cgi-bin/metapathways?db=TriharM10_1&map={map_id}&models=1&col=&batchId=TriharM10_1:1"
    
    archivo = f"API/subsystems/{map_id}_models.html"
    
    if os.path.exists(archivo):
        print(f"⏭️ [{i}/{len(enlaces_modelos)}] Omitiendo {map_id} (models=1) - archivo ya existe.")
        omitidos_subs += 1
        continue
    
    print(f"\n📥 [{i}/{len(enlaces_modelos)}] Descargando subsystems para {map_id} (modelos: {numero})")
    try:
        driver.get(url_subsystem)
        WebDriverWait(driver, 15).until(EC.presence_of_element_located((By.TAG_NAME, "body")))
        time.sleep(3)
        
        html_subs = driver.page_source
        with open(archivo, "w", encoding="utf-8") as f:
            f.write(html_subs)
        print(f"   ✅ Guardado: {archivo}")
        descargados_subs += 1
        time.sleep(1)
    except Exception as e:
        print(f"   ❌ Error al descargar {url_subsystem}: {e}")

print(f"\n✅ Subsystems completados. Descargados: {descargados_subs}, Omitidos: {omitidos_subs}")

# ------------------------------------------------------------------
# 6. PROCESAR HTML DE SUBSYSTEMS: EXTRAER PROTEIN ID Y EC NUM
# ------------------------------------------------------------------
print("\n🔍 Procesando archivos de subsystems para extraer Protein ID y EC Num...")

# Diccionario para almacenar: protein_id -> ec_num (de cada archivo)
protein_ec_map = {}

# Listar todos los archivos HTML en API/subsystems/
subsystem_files = [f for f in os.listdir("API/subsystems") if f.endswith(".html")]
total_archivos = len(subsystem_files)

for idx, filename in enumerate(subsystem_files, 1):
    print(f"\n📄 [{idx}/{total_archivos}] Procesando archivo: {filename}")
    
    filepath = os.path.join("API/subsystems", filename)
    with open(filepath, "r", encoding="utf-8") as f:
        html_content = f.read()
    
    soup = BeautifulSoup(html_content, 'html.parser')
    
    # Buscar la tabla que contiene los datos
    tables = soup.find_all('table')
    if not tables:
        print(f"   ⚠️ No se encontraron tablas en {filename}")
        continue
    
    archivo_protein_ec = {}  # para contar cuántos se extraen de este archivo
    
    for table in tables:
        rows = table.find_all('tr')
        if len(rows) < 2:
            continue
        
        # Buscar encabezados
        header_row = rows[0]
        header_cells = header_row.find_all(['th', 'td'])
        header_texts = [cell.get_text(strip=True) for cell in header_cells]
        
        try:
            idx_protein = header_texts.index("Protein ID")
            idx_ec = header_texts.index("EC Num")
        except ValueError:
            try:
                idx_protein = next(i for i, t in enumerate(header_texts) if "protein" in t.lower())
                idx_ec = next(i for i, t in enumerate(header_texts) if "ec" in t.lower() and "num" in t.lower())
            except StopIteration:
                continue
        
        # Procesar filas de datos
        for row in rows[1:]:
            cells = row.find_all(['th', 'td'])
            if len(cells) <= max(idx_protein, idx_ec):
                continue
            
            protein_id = cells[idx_protein].get_text(strip=True)
            ec_num = cells[idx_ec].get_text(strip=True)
            
            if protein_id and ec_num and re.match(r'^\d+\.\d+\.\d+\.?-?\d*$', ec_num):
                # Guardar en el diccionario principal
                protein_ec_map[protein_id] = ec_num
                # Guardar también en el dict del archivo para contar
                archivo_protein_ec[protein_id] = ec_num
        
        # Si encontramos datos, salir del bucle de tablas
        if archivo_protein_ec:
            break
    
    # Mostrar cuántos se extrajeron de este archivo
    if archivo_protein_ec:
        print(f"   ✅ Extraídos {len(archivo_protein_ec)} pares de este archivo.")
    else:
        print(f"   ℹ️ No se encontraron pares válidos en este archivo.")

print(f"\n✅ Total extraídos: {len(protein_ec_map)} pares Protein ID -> EC Num desde los {total_archivos} archivos procesados.")


# ------------------------------------------------------------------
# 7. (OPCIONAL) CARGAR CSV - YA NO SE USA PARA FILTRAR
# ------------------------------------------------------------------
# Si no quieres cargarlo, puedes comentar o eliminar este bloque
# print("\n📂 Cargando genesModelTemp2.csv...")
# try:
#     import pandas as pd
#     df_genes = pd.read_csv("genesModelTemp2.csv")
#     print(f"✅ CSV cargado con {len(df_genes)} registros.")
# except FileNotFoundError:
#     print("⚠️ No se encontró 'genesModelTemp2.csv', se omite.")
# except Exception as e:
#     print(f"⚠️ Error al leer CSV: {e}")

# ------------------------------------------------------------------
# 8. DESCARGAR PÁGINAS DE EC PARA TODOS LOS PROTEIN ID ENCONTRADOS
# ------------------------------------------------------------------
print(f"\n📥 Descargando páginas de EC para los {len(protein_ec_map)} pares Protein ID → EC Num extraídos...")

# Crear subcarpeta para EC
os.makedirs("API/subsystems/ec", exist_ok=True)

descargados_ec = 0
omitidos_ec = 0

for protein_id, ec_num in protein_ec_map.items():
    # Dividir el EC num en sus componentes (ej. 1.2.3.4 -> ec1=1, ec2=2, ec3=3, ec4=4)
    ec_parts = ec_num.split('.')
    # Asegurar que tenemos al menos 3 partes, y hasta 4
    ec1 = ec_parts[0] if len(ec_parts) > 0 else ''
    ec2 = ec_parts[1] if len(ec_parts) > 1 else ''
    ec3 = ec_parts[2] if len(ec_parts) > 2 else ''
    ec4 = ec_parts[3] if len(ec_parts) > 3 else ''
    
    # Construir la URL de búsqueda KEGG
    url_ec = f"https://mycocosm.jgi.doe.gov/cgi-bin/searchKEGG?field1=EC+Number&ec1={ec1}&ec2={ec2}&ec3={ec3}&ec4={ec4}&species=TriharM10_1&info=KEGG&type=KEGG&batchId=TriharM10_1:1"
    
    # Nombre del archivo: protein_id_ec_num.html
    archivo_ec = f"API/subsystems/ec/{protein_id}_{ec_num}.html"
    
    # Verificar si ya existe
    if os.path.exists(archivo_ec):
        print(f"   ⏭️ Omitiendo EC {ec_num} (Protein {protein_id}) - archivo ya existe.")
        omitidos_ec += 1
        continue
    
    print(f"   📥 Descargando EC {ec_num} para Protein {protein_id}...")
    try:
        driver.get(url_ec)
        WebDriverWait(driver, 15).until(EC.presence_of_element_located((By.TAG_NAME, "body")))
        time.sleep(3)
        
        html_ec = driver.page_source
        with open(archivo_ec, "w", encoding="utf-8") as f:
            f.write(html_ec)
        print(f"      ✅ Guardado: {archivo_ec}")
        descargados_ec += 1
        time.sleep(1)
    except Exception as e:
        print(f"      ❌ Error al descargar {url_ec}: {e}")

print(f"\n✅ EC completados. Descargados: {descargados_ec}, Omitidos: {omitidos_ec}")

# ------------------------------------------------------------------
# 9. RESUMEN FINAL (modificado)
# ------------------------------------------------------------------
print("\n🎉 ¡Descarga completada!")
print(f"   📁 Rutas metabólicas: API/")
print(f"      Descargados: {descargados_rutas}, Omitidos: {omitidos_rutas}")
print(f"   📁 Subsystems (models=1): API/subsystems/")
print(f"      Descargados: {descargados_subs}, Omitidos: {omitidos_subs}")
if 'descargados_ec' in locals():
    print(f"   📁 EC (descargados): API/subsystems/ec/")
    print(f"      Descargados: {descargados_ec}, Omitidos: {omitidos_ec}")

driver.quit()



✅ Página principal guardada en API/ (revisa este archivo para ver si contiene los enlaces).
⚠️ No se encontraron enlaces con 'metapathways?map='. Intentando con 'map='...
🔗 Se encontraron 314 enlaces de rutas metabólicas.
📌 Ejemplos de enlaces encontrados:
   1. Alanine, aspartate and glutamate metabolism -> https://mycocosm.jgi.doe.gov/cgi-bin/metapathways?db=TriharM10_1&map=MAP00250&batchId=TriharM10_1:1
   2. 41 -> https://mycocosm.jgi.doe.gov/cgi-bin/metapathways?db=TriharM10_1&map=MAP00250&batchId=TriharM10_1:1
   3. Arginine and proline metabolism -> https://mycocosm.jgi.doe.gov/cgi-bin/metapathways?db=TriharM10_1&map=MAP00330&batchId=TriharM10_1:1
   4. 104 -> https://mycocosm.jgi.doe.gov/cgi-bin/metapathways?db=TriharM10_1&map=MAP00330&batchId=TriharM10_1:1
   5. Cysteine and methionine metabolism -> https://mycocosm.jgi.doe.gov/cgi-bin/metapathways?db=TriharM10_1&map=MAP00270&batchId=TriharM10_1:1
📊 Resumen:
   - Rutas metabólicas (nombres): 164
   - Enlaces numéricos (modelos

In [ ]:
import os
import re
import pandas as pd

# ------------------------------------------------------------------
# 1. EXTRAER PROTEIN IDs DE LOS ARCHIVOS EN API/subsystems/ec/
# ------------------------------------------------------------------
ec_folder = "API/subsystems/ec"
protein_ids_descargados = set()

if os.path.exists(ec_folder):
    for filename in os.listdir(ec_folder):
        if filename.endswith(".html"):
            # El nombre del archivo es: {ProteinID}_{EC}.html
            # Extraemos el ProteinID (todo antes del primer '_')
            protein_id = filename.split('_')[0]
            if protein_id.isdigit():  # Validar que sea un número
                protein_ids_descargados.add(protein_id)
    print(f"✅ Encontrados {len(protein_ids_descargados)} Protein IDs en archivos HTML.")
else:
    print("❌ La carpeta API/subsystems/ec/ no existe.")
    exit()

# ------------------------------------------------------------------
# 2. CARGAR CSV genesModelTemp2.csv
# ------------------------------------------------------------------
try:
    df_genes = pd.read_csv("genesModelTemp2.csv")
    print(f"✅ CSV cargado con {len(df_genes)} registros.")
    # Asegurar que la columna de Protein ID existe
    if 'ProteinID' not in df_genes.columns:
        # Buscar una columna que contenga "protein" (insensible a mayúsculas)
        for col in df_genes.columns:
            if 'protein' in col.lower():
                df_genes.rename(columns={col: 'ProteinID'}, inplace=True)
                break
        else:
            raise KeyError("No se encontró una columna de Protein ID en el CSV.")
except FileNotFoundError:
    print("❌ No se encontró el archivo 'genesModelTemp2.csv'.")
    exit()
except Exception as e:
    print(f"❌ Error al leer el CSV: {e}")
    exit()

# ------------------------------------------------------------------
# 3. COMPARAR Y ENCONTRAR FALTANTES
# ------------------------------------------------------------------
# Convertir Protein IDs del CSV a string para comparación
protein_ids_csv = set(df_genes['ProteinID'].astype(str).values)

# Encontrar los que están en el CSV pero no en los descargados
faltantes = protein_ids_csv - protein_ids_descargados
presentes = protein_ids_csv & protein_ids_descargados

print(f"\n📊 Estadísticas de comparación:")
print(f"   - Protein IDs en el CSV: {len(protein_ids_csv)}")
print(f"   - Protein IDs descargados: {len(protein_ids_descargados)}")
print(f"   - Coinciden (presentes): {len(presentes)}")
print(f"   - Faltan por descargar: {len(faltantes)}")

# ------------------------------------------------------------------
# 4. MOSTRAR EJEMPLOS DE FALTANTES
# ------------------------------------------------------------------
if faltantes:
    print(f"\n🔍 Ejemplos de Protein IDs NO encontrados (primeros 10):")
    for i, pid in enumerate(sorted(faltantes)[:10], 1):
        print(f"   {i}. {pid}")
    
    # Mostrar también algunos GeneIDs asociados (si están en el CSV)
    # Crear un DataFrame solo con los faltantes
    df_faltantes = df_genes[df_genes['ProteinID'].astype(str).isin(faltantes)]
    print(f"\n📋 Tabla de faltantes (primeras 10 filas):")
    print(df_faltantes.head(10).to_string(index=False))
    
    # Opcional: guardar los faltantes en un CSV
    df_faltantes.to_csv("ProteinIDs_faltantes.csv", index=False)
    print(f"\n💾 Tabla completa guardada en 'ProteinIDs_faltantes.csv'")
else:
    print("\n🎉 ¡Todos los Protein IDs del CSV están descargados!")

print("\n✅ Análisis completado.")

✅ Encontrados 494 Protein IDs en archivos HTML.
✅ CSV cargado con 807 registros.

📊 Estadísticas de comparación:
   - Protein IDs en el CSV: 807
   - Protein IDs descargados: 494
   - Coinciden (presentes): 494
   - Faltan por descargar: 313

🔍 Ejemplos de Protein IDs NO encontrados (primeros 10):
   1. 100580
   2. 102114
   3. 102945
   4. 10572
   5. 10634
   6. 110563
   7. 112521
   8. 122391
   9. 128728
   10. 130215

📋 Tabla de faltantes (primeras 10 filas):
      GeneID  ProteinID
THM10_100580     100580
THM10_102114     102114
THM10_102945     102945
 THM10_10572      10572
 THM10_10634      10634
THM10_110563     110563
THM10_112521     112521
THM10_122391     122391
THM10_128728     128728
THM10_130215     130215

💾 Tabla completa guardada en 'ProteinIDs_faltantes.csv'

✅ Análisis completado.


In [8]:
import os
import re
from bs4 import BeautifulSoup

# ------------------------------------------------------------------
# 1. CONFIGURACIÓN
# ------------------------------------------------------------------
subsystems_folder = "API/subsystems"
protein_ids_a_buscar = ["3721","100580"]  # Cambia por el que quieras

# Si quieres buscar varios:
# protein_ids_a_buscar = ["3721", "100580", "27085"]

# ------------------------------------------------------------------
# 2. FUNCIÓN MEJORADA PARA EXTRAER PROTEIN IDs Y EC
# ------------------------------------------------------------------
def extraer_protein_ids_y_ec(html_content, debug=False):
    """
    Extrae todos los pares (Protein ID, EC Num) de TODAS las tablas del HTML.
    Devuelve un diccionario {protein_id: ec_num}.
    Si debug=True, imprime información de las tablas encontradas.
    """
    soup = BeautifulSoup(html_content, 'html.parser')
    protein_ec_map = {}
    
    # Buscar todas las tablas
    tables = soup.find_all('table')
    if debug:
        print(f"   🔍 Encontradas {len(tables)} tablas en el archivo.")
    
    for table_idx, table in enumerate(tables):
        rows = table.find_all('tr')
        if len(rows) < 2:
            if debug:
                print(f"      Tabla {table_idx+1}: menos de 2 filas, ignorada.")
            continue
        
        # --- 2.1. Intentar identificar encabezados ---
        # Buscar en todas las filas (no solo la primera) para encontrar "Protein ID" y "EC Num"
        idx_protein = None
        idx_ec = None
        
        for row_idx, row in enumerate(rows):
            cells = row.find_all(['th', 'td'])
            cell_texts = [cell.get_text(strip=True) for cell in cells]
            
            # Buscar "Protein ID" en esta fila
            for i, text in enumerate(cell_texts):
                if "protein" in text.lower() and "id" in text.lower():
                    idx_protein = i
                if "ec" in text.lower() and "num" in text.lower():
                    idx_ec = i
            
            if idx_protein is not None and idx_ec is not None:
                if debug:
                    print(f"      Tabla {table_idx+1}: encabezados encontrados en fila {row_idx+1}")
                    print(f"         Protein ID en columna {idx_protein}, EC Num en columna {idx_ec}")
                    print(f"         Textos de la fila: {cell_texts}")
                break  # ya encontramos los índices
        
        # Si no se encontraron los encabezados, intentar por posición (basado en el ejemplo de MAP00010)
        if idx_protein is None or idx_ec is None:
            if debug:
                print(f"      Tabla {table_idx+1}: no se encontraron encabezados exactos.")
                print(f"         Intentando por posición (columna 1 = Protein ID, columna 5 = EC Num)")
            # En los archivos de ejemplo, la estructura es:
            # columna 0: Protein Name, columna 1: Protein ID, columna 5: EC Num
            # Pero para ser flexible, usamos un patrón común
            # Verificamos que la tabla tenga al menos 6 columnas en la primera fila de datos
            for row in rows[1:3]:  # revisar primeras filas de datos
                cells = row.find_all(['th', 'td'])
                if len(cells) >= 6:
                    # Asumir que columna 1 es Protein ID, columna 5 es EC Num
                    idx_protein = 1
                    idx_ec = 5
                    if debug:
                        print(f"         Asignados: Protein ID en columna {idx_protein}, EC Num en columna {idx_ec}")
                    break
        
        if idx_protein is None or idx_ec is None:
            if debug:
                print(f"      Tabla {table_idx+1}: no se pudieron determinar los índices, ignorada.")
            continue
        
        # --- 2.2. Extraer datos de las filas ---
        datos_extraidos = 0
        for row in rows[1:]:  # saltar fila de encabezados (si la primera tenía encabezados)
            cells = row.find_all(['th', 'td'])
            if len(cells) <= max(idx_protein, idx_ec):
                continue
            
            protein_id = cells[idx_protein].get_text(strip=True)
            ec_num = cells[idx_ec].get_text(strip=True)
            
            # Validar que no estén vacíos y que EC num tenga formato válido (ej. 1.2.3.4)
            if protein_id and ec_num and re.match(r'^\d+\.\d+\.\d+\.?-?\d*$', ec_num):
                protein_ec_map[protein_id] = ec_num
                datos_extraidos += 1
        
        if debug and datos_extraidos > 0:
            print(f"      Tabla {table_idx+1}: extraídos {datos_extraidos} pares.")
        
        # Si ya encontramos datos en esta tabla, podemos seguir procesando otras tablas
        # (a veces hay más de una tabla con datos)
    
    return protein_ec_map

# ------------------------------------------------------------------
# 3. RECORRER ARCHIVOS Y BUSCAR
# ------------------------------------------------------------------
resultados = {}  # {protein_id: lista de diccionarios con archivo, map_id, ec_num}

if not os.path.exists(subsystems_folder):
    print(f"❌ La carpeta '{subsystems_folder}' no existe.")
    exit()

archivos = [f for f in os.listdir(subsystems_folder) if f.endswith(".html")]
print(f"🔍 Buscando en {len(archivos)} archivos HTML...")

# Activar depuración para ver qué está pasando
# Puedes poner DEBUG = True para ver detalles de cada archivo, o False para solo resultados
DEBUG = True

for filename in archivos:
    filepath = os.path.join(subsystems_folder, filename)
    
    if DEBUG:
        print(f"\n📄 Procesando: {filename}")
    
    with open(filepath, "r", encoding="utf-8") as f:
        html = f.read()
    
    protein_ec = extraer_protein_ids_y_ec(html, debug=DEBUG)
    
    # Extraer MAP ID del nombre del archivo
    map_match = re.match(r'(MAP\d+)_', filename)
    map_id = map_match.group(1) if map_match else filename
    
    # Verificar si alguno de los IDs buscados está en este archivo
    for pid in protein_ids_a_buscar:
        if pid in protein_ec:
            if pid not in resultados:
                resultados[pid] = []
            resultados[pid].append({
                "archivo": filename,
                "map_id": map_id,
                "ec_num": protein_ec[pid]
            })
            if DEBUG:
                print(f"   ✅ ¡ENCONTRADO! {pid} en {filename} con EC {protein_ec[pid]}")

# ------------------------------------------------------------------
# 4. MOSTRAR RESULTADOS
# ------------------------------------------------------------------
print("\n" + "="*60)
print("📊 RESULTADOS DE LA BÚSQUEDA")
print("="*60)

if not resultados:
    print("   ❌ Ninguno de los Protein IDs buscados fue encontrado.")
    print("\n🔍 Sugerencias para depurar:")
    print("   1. Verifica que el Protein ID exista en los archivos (abre uno manualmente).")
    print("   2. Comprueba que la columna se llame 'Protein ID' o 'protein id'.")
    print("   3. Si la tabla tiene otra estructura, ajusta los índices en el código.")
    print("   4. Activa DEBUG=True para ver cómo se procesa cada archivo.")
else:
    for pid, info in resultados.items():
        print(f"\n🔹 Protein ID: {pid}")
        print(f"   Encontrado en {len(info)} archivo(s):")
        for item in info:
            print(f"      - {item['map_id']} ({item['archivo']}) → EC: {item['ec_num']}")
    
    no_encontrados = set(protein_ids_a_buscar) - set(resultados.keys())
    if no_encontrados:
        print(f"\n⚠️ Protein IDs NO encontrados: {', '.join(no_encontrados)}")
        print("   Revisa que estos IDs existan en los archivos HTML.")

# ------------------------------------------------------------------
# 5. GUARDAR REPORTE
# ------------------------------------------------------------------
with open("reporte_busqueda_protein_ids.txt", "w", encoding="utf-8") as f:
    f.write("REPORTE DE BÚSQUEDA DE PROTEIN IDs EN subsystems/\n")
    f.write("="*60 + "\n")
    if not resultados:
        f.write("No se encontraron los Protein IDs buscados.\n")
    else:
        for pid, info in resultados.items():
            f.write(f"\nProtein ID: {pid}\n")
            f.write(f"  Encontrado en {len(info)} archivo(s):\n")
            for item in info:
                f.write(f"    - {item['map_id']} ({item['archivo']}) → EC: {item['ec_num']}\n")
    no_encontrados = set(protein_ids_a_buscar) - set(resultados.keys())
    if no_encontrados:
        f.write(f"\nProtein IDs NO encontrados: {', '.join(no_encontrados)}\n")

print("\n💾 Reporte guardado en 'reporte_busqueda_protein_ids.txt'")

🔍 Buscando en 150 archivos HTML...

📄 Procesando: MAP00010_models.html
   🔍 Encontradas 3 tablas en el archivo.
      Tabla 1: menos de 2 filas, ignorada.
      Tabla 2: encabezados encontrados en fila 1
         Protein ID en columna 3, EC Num en columna 7
         Textos de la fila: ['Protein NameProtein IDModel SetE-valueTop KEGG HitEC NumEnzymeCurated?CE27084_1552127085Trichoderma harzianum M10 v1.0/FilteredModels1 (ver 1)0tre:TRIREDRAFT_21836 K01835 (EC:5.4.2.2)5.4.2.2UnknownNOCE38990_3344738991Trichoderma harzianum M10 v1.0/FilteredModels1 (ver 1)0tre:TRIREDRAFT_73774 K01895 (EC:6.2.1.1)6.2.1.1UnknownNOCE39774_3683639775Trichoderma harzianum M10 v1.0/FilteredModels1 (ver 1)0nhe:NECHADRAFT_70106 K00627 (EC:2.3.1.12)2.3.1.12UnknownNOCE45769_27545770Trichoderma harzianum M10 v1.0/FilteredModels1 (ver 1)0ctp:CTRG_05482 K13953 (EC:1.1.1.1)1.1.1.1UnknownNOCE104330_1157104331Trichoderma harzianum M10 v1.0/FilteredModels1 (ver 1)0tre:TRIREDRAFT_121661 K01785 (EC:5.1.3.3)5.1.3.3UnknownNOC

In [14]:
import os
import glob
import re
import pandas as pd
from bs4 import BeautifulSoup

# ------------------------------------------------------------
# 1. LEER CSV
# ------------------------------------------------------------
try:
    df_genes = pd.read_csv("genesModelTemp2.csv")
    print(f"✅ CSV cargado: {len(df_genes)} registros.")
except FileNotFoundError:
    print("❌ No se encontró 'genesModelTemp2.csv'.")
    exit()
except Exception as e:
    print(f"❌ Error: {e}")
    exit()

if 'ProteinID' not in df_genes.columns:
    for col in df_genes.columns:
        if 'protein' in col.lower():
            df_genes.rename(columns={col: 'ProteinID'}, inplace=True)
            break
    else:
        print("❌ Columna ProteinID no encontrada.")
        exit()

protein_ids = set(df_genes['ProteinID'].astype(str).values)
print(f"🔍 Buscando {len(protein_ids)} Protein IDs.")

# ------------------------------------------------------------
# 2. FUNCIÓN DE EXTRACCIÓN MEJORADA
# ------------------------------------------------------------
def extraer_datos_ec(html_path, debug=False):
    try:
        with open(html_path, 'r', encoding='utf-8') as f:
            html = f.read()
    except:
        return None

    soup = BeautifulSoup(html, 'html.parser')
    
    for table in soup.find_all('table'):
        rows = table.find_all('tr')
        if len(rows) < 1:
            continue
        
        # Caso 1: la primera fila tiene una sola celda con todo pegado
        first_row_cells = rows[0].find_all(['th', 'td'])
        if len(first_row_cells) == 1:
            cell_text = first_row_cells[0].get_text(strip=True)
            # Verificar si contiene EC Number y Definition pegados
            if 'EC Number' in cell_text and 'Definition' in cell_text:
                # Dividir usando los encabezados como separadores
                # La cadena empieza con encabezados pegados: EC NumberDefinitionAlternative Name...
                # Luego vienen los valores correspondientes
                # Podemos usar re.split con los nombres de los campos
                pattern = r'(EC Number|Definition|Alternative Name|Catalytic Activity|Cofactors|Associated Diseases)'
                parts = re.split(pattern, cell_text)
                # parts tendrá ['', 'EC Number', 'valor', 'Definition', 'valor', ...]
                ec_number = ''
                definition = ''
                catalytic = ''
                # Buscar los valores
                for i in range(1, len(parts)-1, 2):
                    key = parts[i].strip()
                    value = parts[i+1].strip()
                    if key == 'EC Number':
                        ec_number = value
                    elif key == 'Definition':
                        definition = value
                    elif key == 'Catalytic Activity':
                        catalytic = value
                if ec_number:
                    return {
                        'EC_Number': ec_number,
                        'Definition': definition,
                        'Catalytic_Activity': catalytic
                    }
                # Si no se encontró, intentar con el método alternativo
                # Tal vez los valores están al final
                # Extraer usando patrones más simples
                # Buscar EC number (ej: 1.14.19.-) al final de la cadena
                # ...
        
        # Caso 2: tabla con filas de encabezado y datos separadas
        # Buscar fila que contenga "EC Number" y "Definition" en celdas separadas
        for row_idx, row in enumerate(rows):
            cells = row.find_all(['th', 'td'])
            cell_texts = [cell.get_text(strip=True) for cell in cells]
            if any('EC Number' in text for text in cell_texts) and any('Definition' in text for text in cell_texts):
                # Esta es la fila de encabezados, la siguiente fila contiene los datos
                if row_idx + 1 < len(rows):
                    data_row = rows[row_idx + 1]
                    data_cells = data_row.find_all(['th', 'td'])
                    if len(data_cells) >= 4:
                        # Extraer por posición (0: EC, 1: Definition, 3: Catalytic Activity)
                        ec_number = data_cells[0].get_text(strip=True)
                        definition = data_cells[1].get_text(strip=True)
                        catalytic = data_cells[3].get_text(strip=True) if len(data_cells) > 3 else ''
                        if ec_number and ec_number != 'EC Number':
                            return {
                                'EC_Number': ec_number,
                                'Definition': definition,
                                'Catalytic_Activity': catalytic
                            }
        # Si no se encontró, probar con pandas
        try:
            tables = pd.read_html(html)
            for df in tables:
                if df.shape[0] > 1:
                    for idx, row in df.iterrows():
                        row_text = row.astype(str).values
                        if any('EC Number' in val for val in row_text) and any('Definition' in val for val in row_text):
                            if idx + 1 < df.shape[0]:
                                data_row = df.iloc[idx + 1]
                                ec_num = data_row.iloc[0] if len(data_row) > 0 else ''
                                definition = data_row.iloc[1] if len(data_row) > 1 else ''
                                catalytic = data_row.iloc[3] if len(data_row) > 3 else ''
                                return {
                                    'EC_Number': str(ec_num),
                                    'Definition': str(definition),
                                    'Catalytic_Activity': str(catalytic)
                                }
        except Exception:
            pass
    return None

# ------------------------------------------------------------
# 3. PROCESAR PROTEIN IDs
# ------------------------------------------------------------
ec_folder = "API/subsystems/ec"
if not os.path.exists(ec_folder):
    print(f"❌ Carpeta '{ec_folder}' no existe.")
    exit()

archivos_html = glob.glob(os.path.join(ec_folder, "*.html"))
print(f"📁 Encontrados {len(archivos_html)} archivos HTML.")

protein_to_file = {}
for f in archivos_html:
    basename = os.path.basename(f)
    parts = basename.split('_')
    if len(parts) >= 2:
        pid = parts[0]
        protein_to_file[pid] = f
    else:
        pid = basename.replace('.html', '')
        protein_to_file[pid] = f

print(f"🔗 Mapeados {len(protein_to_file)} Protein IDs.")

resultados = []
for pid in protein_ids:
    if pid in protein_to_file:
        archivo = protein_to_file[pid]
        datos = extraer_datos_ec(archivo)
        if datos:
            resultados.append({
                'ProteinID': pid,
                'EC_Number': datos['EC_Number'],
                'Definition': datos['Definition'],
                'Catalytic_Activity': datos['Catalytic_Activity'],
                'Archivo': os.path.basename(archivo)
            })
        else:
            resultados.append({
                'ProteinID': pid,
                'EC_Number': 'No encontrado',
                'Definition': 'No encontrado',
                'Catalytic_Activity': 'No encontrado',
                'Archivo': os.path.basename(archivo)
            })
    else:
        resultados.append({
            'ProteinID': pid,
            'EC_Number': 'No hay archivo',
            'Definition': 'No hay archivo',
            'Catalytic_Activity': 'No hay archivo',
            'Archivo': 'No existe'
        })

# ------------------------------------------------------------
# 4. GUARDAR Y MOSTRAR
# ------------------------------------------------------------
df_resultado = pd.DataFrame(resultados)
print("\n📋 Vista previa (primeros 10):")
print(df_resultado.head(10).to_string(index=False))

df_resultado.to_csv("ProteinID_EC_info.csv", index=False, encoding='utf-8-sig')
print("\n💾 Guardado en 'ProteinID_EC_info.csv'")

✅ CSV cargado: 807 registros.
🔍 Buscando 807 Protein IDs.
📁 Encontrados 494 archivos HTML.
🔗 Mapeados 494 Protein IDs.

📋 Vista previa (primeros 10):
ProteinID      EC_Number                                                  Definition                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                              

In [ ]:
import csv
import re

def extract_protein_id(header):
    """
    Extrae el número que sigue al guion bajo en el encabezado.
    'THM10_100473' -> '100473'
    """
    parts = header.split('_')
    if len(parts) > 1:
        return parts[1] 
    return ''

# Leer el archivo FASTA y escribir el CSV
with open('THM10.fasta', 'r') as fasta_file, \
     open('THM10_info.csv', 'w', newline='') as csv_file:

    writer = csv.writer(csv_file)
    writer.writerow(['GeneID', 'ProteinID'])  # encabezados

    for line in fasta_file:
        line = line.strip()
        if not line:
            continue
        if line.startswith('>'):
            header = line[1:]  # quitar el '>'
            gene_id = header
            protein_id = extract_protein_id(header)
            writer.writerow([gene_id, protein_id])

print("Archivo THM10_info.csv generado correctamente.")

Archivo THM10_info.csv generado correctamente.


In [3]:
import os
import re
import pandas as pd
from bs4 import BeautifulSoup
from pathlib import Path

# ------------------------------------------------------------
# 1. LEER CSV
# ------------------------------------------------------------
csv_path = "THM10_info.csv"
if not os.path.exists(csv_path):
    print(f"❌ No se encontró '{csv_path}'.")
    exit()

df_genes = pd.read_csv(csv_path)
print(f"✅ CSV cargado: {len(df_genes)} registros.")

col_protein = None
for col in df_genes.columns:
    if 'protein' in col.lower():
        col_protein = col
        break
if col_protein is None:
    print("❌ No se encontró columna con 'protein' en el nombre.")
    exit()
else:
    if col_protein != 'ProteinID':
        df_genes.rename(columns={col_protein: 'ProteinID'}, inplace=True)

protein_ids = set(df_genes['ProteinID'].astype(str).values)
print(f"🔍 Buscando {len(protein_ids)} Protein IDs.")

# ------------------------------------------------------------
# 2. PROCESAR ARCHIVOS HTML
# ------------------------------------------------------------
ec_folder = Path("API/subsystems/ec")
if not ec_folder.exists():
    print(f"❌ Carpeta '{ec_folder}' no existe.")
    exit()

archivos_html = list(ec_folder.glob("*.html"))
print(f"📁 Encontrados {len(archivos_html)} archivos HTML.")

protein_to_file = {}
for f in archivos_html:
    basename = f.name
    parts = basename.split('_')
    if len(parts) >= 2:
        pid = parts[0]
    else:
        pid = basename.replace('.html', '')
    protein_to_file[pid] = f

print(f"🔗 Mapeados {len(protein_to_file)} Protein IDs.")

# ------------------------------------------------------------
# 3. PROCESAR CADA PROTEIN ID
# ------------------------------------------------------------
resultados = []
for pid in protein_ids:
    resultado = {
        'ProteinID': pid,
        'EC_Number': '',
        'Definition': '',
        'Catalytic_Activity': '',
        'Archivo': '',
        'KEGG': ''
    }
    
    if pid not in protein_to_file:
        resultado['EC_Number'] = 'No hay archivo'
        resultado['Definition'] = 'No hay archivo'
        resultado['Catalytic_Activity'] = 'No hay archivo'
        resultado['Archivo'] = 'No existe'
        resultados.append(resultado)
        continue
    
    archivo = protein_to_file[pid]
    resultado['Archivo'] = archivo.name
    
    try:
        with open(archivo, 'r', encoding='utf-8') as f:
            html = f.read()
    except Exception as e:
        print(f"⚠️ Error leyendo {archivo.name}: {e}")
        resultado['EC_Number'] = 'Error lectura'
        resultado['Definition'] = 'Error lectura'
        resultado['Catalytic_Activity'] = 'Error lectura'
        resultados.append(resultado)
        continue
    
    soup = BeautifulSoup(html, 'html.parser')
    encontrado = False
    
    # --- Extraer de tablas ---
    tables = soup.find_all('table')
    for table in tables:
        rows = table.find_all('tr')
        if len(rows) < 1:
            continue
        
        for row_idx, row in enumerate(rows):
            cells = row.find_all(['th', 'td'])
            cell_texts = [cell.get_text(strip=True) for cell in cells]
            if any('EC Number' in text for text in cell_texts) and any('Definition' in text for text in cell_texts):
                if row_idx + 1 < len(rows):
                    data_row = rows[row_idx + 1]
                    data_cells = data_row.find_all(['th', 'td'])
                    if len(data_cells) >= 2:
                        ec_num = data_cells[0].get_text(strip=True)
                        definition = data_cells[1].get_text(strip=True)
                        catalytic = ''
                        if len(data_cells) > 3:
                            catalytic = data_cells[3].get_text(strip=True)
                        if ec_num and ec_num != 'EC Number':
                            resultado['EC_Number'] = ec_num
                            resultado['Definition'] = definition
                            resultado['Catalytic_Activity'] = catalytic
                            encontrado = True
                            break
        if encontrado:
            break
        
        if not encontrado:
            first_row = rows[0]
            cells = first_row.find_all(['th', 'td'])
            if len(cells) == 1:
                cell_text = cells[0].get_text(strip=True)
                ec_match = re.search(r'EC Number\s*([^\s]+)', cell_text, re.IGNORECASE)
                if ec_match:
                    resultado['EC_Number'] = ec_match.group(1)
                    def_match = re.search(r'Definition\s*([^C]+?)(?=Catalytic Activity|$)', cell_text, re.IGNORECASE)
                    if def_match:
                        resultado['Definition'] = def_match.group(1).strip()
                    cat_match = re.search(r'Catalytic Activity\s*([^C]+?)(?=EC Number|$)', cell_text, re.IGNORECASE)
                    if cat_match:
                        resultado['Catalytic_Activity'] = cat_match.group(1).strip()
                    encontrado = True
                    break
    
    # Respaldo con pandas
    if not encontrado:
        try:
            tables_pd = pd.read_html(html)
            for df in tables_pd:
                if df.shape[0] > 1:
                    for idx, row in df.iterrows():
                        row_text = row.astype(str).values
                        if any('EC Number' in val for val in row_text) and any('Definition' in val for val in row_text):
                            if idx + 1 < df.shape[0]:
                                data_row = df.iloc[idx + 1]
                                ec_num = str(data_row.iloc[0]) if len(data_row) > 0 else ''
                                definition = str(data_row.iloc[1]) if len(data_row) > 1 else ''
                                catalytic = str(data_row.iloc[3]) if len(data_row) > 3 else ''
                                if ec_num and ec_num != 'EC Number':
                                    resultado['EC_Number'] = ec_num
                                    resultado['Definition'] = definition
                                    resultado['Catalytic_Activity'] = catalytic
                                    encontrado = True
                                    break
                if encontrado:
                    break
        except Exception:
            pass
    
    if not encontrado:
        resultado['EC_Number'] = 'No encontrado'
        resultado['Definition'] = 'No encontrado'
        resultado['Catalytic_Activity'] = 'No encontrado'
    
    # ------------------------------------------------------------
    # EXTRACCIÓN DE KEGG (prioriza Catalytic_Activity)
    # ------------------------------------------------------------
    def extraer_kegg(texto):
        if not texto or texto in ['No hay archivo', 'Error lectura', 'No encontrado']:
            return ''
        # Limpiar texto
        texto = re.sub(r'\s+', ' ', texto).strip()
        patrones = [
            r'\[RN\s*:\s*([Rr]\d+)\]',   # [RN:R00897]
            r'RN\s*:\s*([Rr]\d+)',        # RN:R00897
            r'\[RN\s*([Rr]\d+)\]',        # [RN R00897]
            r'RN\s*([Rr]\d+)'             # RN R00897
        ]
        for patron in patrones:
            match = re.search(patron, texto)
            if match:
                return match.group(1)
        return ''

    # Buscar en Catalytic_Activity primero, luego en Definition, luego en todo el HTML
    kegg = extraer_kegg(resultado['Catalytic_Activity'])
    if not kegg:
        kegg = extraer_kegg(resultado['Definition'])
    if not kegg:
        kegg = extraer_kegg(html)
    resultado['KEGG'] = kegg

    resultados.append(resultado)

# ------------------------------------------------------------
# 4. GUARDAR Y MOSTRAR
# ------------------------------------------------------------
df_resultado = pd.DataFrame(resultados)
print("\n📋 Vista previa (primeros 10):")
print(df_resultado.head(10).to_string(index=False))

df_resultado.to_csv("THM10_EC_info.csv", index=False, encoding='utf-8-sig')
print("\n💾 Guardado en 'THM10_EC_info.csv'")

✅ CSV cargado: 12842 registros.
🔍 Buscando 12842 Protein IDs.
📁 Encontrados 1972 archivos HTML.
🔗 Mapeados 1972 Protein IDs.

📋 Vista previa (primeros 10):
ProteinID      EC_Number        Definition                                                      Catalytic_Activity              Archivo   KEGG
   360198 No hay archivo    No hay archivo                                                          No hay archivo            No existe       
   430938 No hay archivo    No hay archivo                                                          No hay archivo            No existe       
   498453 No hay archivo    No hay archivo                                                          No hay archivo            No existe       
    31958 No hay archivo    No hay archivo                                                          No hay archivo            No existe       
   371020 No hay archivo    No hay archivo                                                          No hay archivo            No 